# Bayesian Weibull survival model

This notebook fits the minimal non-hierarchical, right-censored Bayesian Weibull model on a bounded sample. It reports posterior summaries and compares the model-implied median with the Kaplan-Meier median from the same rows.

The smoke settings below are intentionally small. They are for a reproducible pipeline check, not a final convergence claim. Hierarchical fitting and generic classical-comparison helpers remain deferred.

In [ ]:
from pathlib import Path
import sys

import arviz as az
import duckdb
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.bayesian_survival import fit_bayesian_weibull
from src.survival_classical import fit_kaplan_meier

PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cases_clean.parquet'
ROWS = 250
DRAWS = 80
TUNE = 80
RANDOM_SEED = 11
if not PROCESSED_PATH.exists():
    raise FileNotFoundError(PROCESSED_PATH)
print({'processed_path': str(PROCESSED_PATH), 'rows': ROWS, 'draws': DRAWS, 'tune': TUNE})


## Bounded same-sample cohort

This selects 250 cases from state `01` in 2018 through DuckDB. The full parquet is not loaded into pandas.

In [ ]:
query = """
SELECT duration, event
FROM read_parquet(?)
WHERE state = '01' AND year = 2018
ORDER BY ddl_case_id
LIMIT ?
"""
with duckdb.connect() as con:
    sample = con.execute(query, [str(PROCESSED_PATH), ROWS]).fetchdf()

print({
    'rows': len(sample),
    'events': int(sample['event'].sum()),
    'censored': int((sample['event'] == 0).sum()),
})


## Posterior fit and summaries

Observed cases contribute the Weibull density. Pending cases contribute the survival probability at their censoring time.

In [ ]:
idata = fit_bayesian_weibull(
    sample, draws=DRAWS, tune=TUNE, random_seed=RANDOM_SEED
)
posterior_summary = az.summary(idata, var_names=['shape', 'scale'], hdi_prob=0.94)
display(posterior_summary)


## Bayesian median versus Kaplan-Meier median

For a Weibull distribution, the median is `scale * log(2) ** (1 / shape)`. This compares that posterior distribution with the nonparametric KM median on the identical bounded cohort.

In [ ]:
posterior_shape = idata.posterior['shape']
posterior_scale = idata.posterior['scale']
posterior_median = posterior_scale * np.log(2) ** (1 / posterior_shape)
bayesian_median = {
    'mean_days': float(posterior_median.mean()),
    'hdi_3_percent_days': float(posterior_median.quantile(0.03)),
    'hdi_97_percent_days': float(posterior_median.quantile(0.97)),
}
km_fit = fit_kaplan_meier(sample)
comparison = pd.DataFrame([
    {'method': 'Bayesian Weibull posterior median', 'median_days': bayesian_median['mean_days']},
    {'method': 'Kaplan-Meier median', 'median_days': float(km_fit.median_survival_time_)},
])
print({'bayesian_median_summary': bayesian_median})
display(comparison)


## Interpretation boundary

This is a bounded smoke analysis with only 80 posterior draws and 80 tuning steps. Use it to verify the model path, not to report final credible intervals or substantive state findings. A production comparison needs longer sampling, convergence checks, and a prespecified cohort.